## 📊 Bolsa Família (São Paulo): Socioeconomic Data Pipeline & Analysis

#### 🎯 Problema de Negócio: Como os recursos de assistência social estão distribuídos dentro do estado mais rico do Brasil? Existem bolsões de pobreza escondidos no interior ou o capital está concentrado na região metropolitana?

In [ ]:
import pandas as pd
import glob

# Passo 1: Juntar todos os arquivos em 1 arquivo apenas
    # Passo 1.1: Gerar DF apenas do estado de SP

# Mapeia todos os arquivos .csv da pasta
caminho_arquivo = glob.glob('C:/caminho/*.csv')

# Lista vazia para guardar os mês de SP
lista_dfs_sp = []

# Inicia o loop, vai ler cada arquivo, filtrar e guardar um arquivo por vez
for arquivo in caminho_arquivo:
    print(f'Processando arquivo: {arquivo}')

    # Lê 1 arquivo por vez
    df_mes_completo = pd.read_csv(arquivo, sep=';', encoding='latin-1')

    # Filtra apenas o estado de SP
    df_mes_sp = df_mes_completo[df_mes_completo['UF'] == 'SP']

    # Guarda o df filtrado na lista
    lista_dfs_sp.append(df_mes_sp)

# Fora do loop, concatena todos os dfs de São Paulo em um único df
df_sp = pd.concat(lista_dfs_sp, ignore_index=True)

print('Concatenação concluída com sucecsso.')
display(df_mes_sp.sample(10)) # Exibe 10 linhas aleatórias do df
display(df_sp.info()) # Exibe todas as colunas e seus tipos de dados


In [ ]:
# Passo 2: Tratar os dados (inuteis, nulos, formatos)

# Soma quantos valores nulos tem em cada coluna
display(df_sp.isna().sum())

# Removemos as colunas que não iremos usar
df_sp = df_sp.drop(columns=['CÓDIGO MUNICÍPIO SIAFI', 'CPF FAVORECIDO'])

# Removemos as linhas onde o NIS está NaN (nulo/sem valor)
df_sp = df_sp.dropna()

# Remove espaços desnecessários antes e depois do texto, e deixa em CAIXA ALTA
df_sp['NOME MUNICÍPIO'] = df_sp['NOME MUNICÍPIO'].str.strip().str.upper()
df_sp['NOME FAVORECIDO'] = df_sp['NOME FAVORECIDO'].str.strip().str.upper()

# Transformamos o campo do NIS em str e removemos os '.' do texto
df_sp['NIS FAVORECIDO'] = df_sp['NIS FAVORECIDO'].astype('str')
df_sp['NIS FAVORECIDO'] = df_sp['NIS FAVORECIDO'].str.replace('.', '', regex=False)

# Trocando a vírugla do decimal por ponto (',', '.')
df_sp['VALOR PARCELA'] = df_sp['VALOR PARCELA'].str.replace(',', '.', regex=False)
# Convertemos o campo VALOR PARCELA de str para float, depois de tratado a vírgula
df_sp['VALOR PARCELA'] = df_sp['VALOR PARCELA'].astype('Float64')

# Ordenando a tabela pelo nome da cidade e nome do favorecido
df_sp = df_sp.sort_values(by=['NOME MUNICÍPIO', 'NOME FAVORECIDO'])
# Reseta a coluna de index, porem, vira uma nova coluna. Solução já é dropar essa coluna
df_sp = df_sp.reset_index(drop=True)

display(df_sp)
display(df_sp.info()) # Exibe todas as colunas e seus tipos de dados
display(df_sp.isnull().sum()) # Soma novamentes quantos valores nulos tem em cada coluna

MÊS COMPETÊNCIA                 0
MÊS REFERÊNCIA                  0
UF                              0
CÓDIGO MUNICÍPIO SIAFI          0
NOME MUNICÍPIO                  0
CPF FAVORECIDO            9739903
NIS FAVORECIDO                516
NOME FAVORECIDO                 0
VALOR PARCELA                   0
dtype: int64

,MÊS COMPETÊNCIA,MÊS REFERÊNCIA,UF,NOME MUNICÍPIO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
0,202501,202501,SP,ADAMANTINA,160924295790,ABDIAS FERNANDES DA SILVA,600.0
1,202502,202502,SP,ADAMANTINA,160924295790,ABDIAS FERNANDES DA SILVA,600.0
2,202503,202503,SP,ADAMANTINA,160924295790,ABDIAS FERNANDES DA SILVA,600.0
3,202504,202504,SP,ADAMANTINA,160924295790,ABDIAS FERNANDES DA SILVA,600.0
4,202505,202505,SP,ADAMANTINA,160924295790,ABDIAS FERNANDES DA SILVA,600.0
...,...,...,...,...,...,...,...
34267097,202511,202511,SP,ZACARIAS,213849920320,YASMIM PISTOLATO MECIANO,900.0
34267098,202512,202512,SP,ZACARIAS,213849920320,YASMIM PISTOLATO MECIANO,900.0
34267099,202601,202601,SP,ZACARIAS,213849920320,YASMIM PISTOLATO MECIANO,900.0
34267100,202602,202602,SP,ZACARIAS,213849920320,YASMIM PISTOLATO MECIANO,900.0


<class 'pandas.DataFrame'>
RangeIndex: 34267102 entries, 0 to 34267101
Data columns (total 7 columns):
 #   Column           Dtype  
---  ------           -----  
 0   MÊS COMPETÊNCIA  int64  
 1   MÊS REFERÊNCIA   int64  
 2   UF               str    
 3   NOME MUNICÍPIO   str    
 4   NIS FAVORECIDO   str    
 5   NOME FAVORECIDO  str    
 6   VALOR PARCELA    Float64
dtypes: Float64(1), int64(2), str(4)
memory usage: 3.4 GB


None

MÊS COMPETÊNCIA    0
MÊS REFERÊNCIA     0
UF                 0
NOME MUNICÍPIO     0
NIS FAVORECIDO     0
NOME FAVORECIDO    0
VALOR PARCELA      0
dtype: int64

In [ ]:
# Passo 3: Análisar os dados

# 3.1. Volume Total por Município
# Criamos um df com as colunas ['NOME MUNICÍPIO', 'VALOR PARCELA'], onde será somada o valor da parela, e agrupado por município
# .sorte_values() - estamos ordenando o df pelo valor total, de forma decrescente
volume_tot = df_sp[['NOME MUNICÍPIO', 'VALOR PARCELA']].groupby('NOME MUNICÍPIO').sum().sort_values('VALOR PARCELA', ascending=False)
volume_tot = volume_tot.reset_index() # resta o index do df

print('TOP 10 CIDADES POR VOLUME TOTAL (R$)')
display(volume_tot.head(10)) # Exibe as primeiras 10 linhas do df
print('=='*60)

# 3.2. Ticket Médio por Município
# Primeiro agrupamos por cidade e mês, para não misturar os pagamentos de meses diferentes
ticket_medio = df_sp.groupby(['NOME MUNICÍPIO', 'MÊS COMPETÊNCIA']).agg(
    VOLUME_TOT = ('VALOR PARCELA', 'sum'),              # Soma de todos os valores pagos
    QTD_PARCELAS = ('VALOR PARCELA', 'count'),          # Conta quantas parcelas foram pagas
    QTD_FAVORECIDOS = ('NIS FAVORECIDO', 'nunique')     # Conta quantas familias únicas tem (conta NIS FAVORECIDOS distintos)
).reset_index()

# Calcumaos o valor médio que cada família recebeu naquele mês específico
ticket_medio['TICKET MEDIO'] = ticket_medio['VOLUME_TOT'] / ticket_medio['QTD_FAVORECIDOS']

# Filtramos as cidades com mais de 1000 favorecidos (para evitar distorções estatísticas de cidades minúsculas)
ticket_medio = ticket_medio[ticket_medio['QTD_FAVORECIDOS'] >= 1000]

# Agora que temos a média de cada mês, tiramos a média novamente por cidade
# Isso nos dá o valor real que a cidade paga por mês, considerando todo o período
ticket_medio = ticket_medio.groupby('NOME MUNICÍPIO').agg(
    TICKET_MEDIO = ('TICKET MEDIO', 'mean')
).sort_values(by='TICKET_MEDIO', ascending=False).reset_index()

# Ordenamos o df pelo Ticket Medio e resetamos/removemos o índice
#ticket_medio = ticket_medio.sort_values(by='TICKET MEDIO', ascending=False)
#icket_medio = ticket_medio.reset_index(drop=True)

print('TICKET MÉDIO POR CIDADE (R$)')
display(ticket_medio)
print('=='*60)

# 3.3. Proporção Capital vs Interior
# Criamos uma lista das cidadeds da região metropoliatana de São Paulo, para separar do interior do estado
cidades_sp = [
    'GUARULHOS', 'OSASCO', 'SUZANO', 'SANTO ANDRE', 'SAO BERNARDO DO CAMPO', 'SAO CAETANO DO SUL', 'DIADEMA','MAUA',
    'MOGI DAS CRUZES', 'ITAQUAQUECETUBA', 'TABOAO DA SERRA', 'BARUERI', 'CARAPICUIBA', 'COTIA', 'EMBU DAS ARTES',
    'FRANCO DA ROCHA', 'CAIEIRAS', 'FERRAZ DE VASCONCELOS', 'POA', 'RIBEIRAO PIRES', 'SAO PAULO'
]

# Calculamos o volume total do estado
total_estado = df_sp['VALOR PARCELA'].sum()

# Calculamos o volume total para São Paulo e Região Metropolitana
# .isin está analisando se o 'NOME MUNICÍPIO' atual está dentro da lista criada acima (cidades_sp)
total_capital = df_sp[df_sp['NOME MUNICÍPIO'].isin(cidades_sp)]['VALOR PARCELA'].sum()

# Calculamos o volume total do interior (demais cidades fora da lista acima "cidades_sp")
total_interior = total_estado - total_capital

# Calculamos o percentual de cada sobre o volume total
pct_capital = (total_capital / total_estado) * 100
pct_interior = (total_interior / total_estado) * 100

print(f'--- DISTRIBUICAO: CAPITAL vs INTERIOR ---')
print(f'Total do estado: {total_estado:.2f}') # :.2f formata o valor para 2 casas decimais (ex: 123.1234 => 123.12)
print(f'Capital (São Paulo e Região Metropolitana): {pct_capital:.2f}% (R$ {total_capital:.2f})')
print(f'Interior (Demais cidades): {pct_interior:.2f}% (R${total_interior:.2f})')


TOP 10 CIDADES POR VOLUME TOTAL (R$)


,NOME MUNICÍPIO,VALOR PARCELA
0,SAO PAULO,6279736146.0
1,GUARULHOS,1018306236.0
2,CAMPINAS,537111028.0
3,ITAQUAQUECETUBA,374742066.0
4,OSASCO,371081253.0
5,SAO BERNARDO DO CAMPO,354929190.0
6,SANTO ANDRE,345593273.0
7,SAO JOSE DOS CAMPOS,333998925.0
8,MOGI DAS CRUZES,330858688.0
9,SOROCABA,307184270.0


TICKET MÉDIO POR CIDADE


,NOME MUNICÍPIO,TICKET_MEDIO
0,VARGEM GRANDE DO SUL,715.52998
1,SAO VICENTE,711.434406
2,PRAIA GRANDE,708.340324
3,JARDINOPOLIS,702.823813
4,BRODOWSKI,701.043525
...,...,...
269,JUNQUEIROPOLIS,633.037348
270,PIRACAIA,631.29315
271,NOVO HORIZONTE,624.36867
272,OSVALDO CRUZ,622.025248


--- DISTRIBUICAO: CAPITAL vs INTERIOR ---
Total do estado: 22621120589.00
Capital (São Paulo e Região Metropolitana): 48.90% (R$ 11061265057.00)
Interior (Demais cidades): 51.10% (R$11559855532.00)


In [ ]:
# Passo 4: Exportar o arquivo em formato de Banco de Dados, deixando mais leve para usar no Power Bi
from sqlalchemy import create_engine
# 4.1. Criamos o "motor" de conexão
# Se o arquivo não existir, o Python cria ele na hora.
engine = create_engine('sqlite:///C:/caminho/bolsa_familia_sp_clean.db')

# 4.2. Usamos o to_sql para enviar o DataFrame para o banco
# 'bolsa_familia_sp_clean' será o nome da tabela dentro do banco
df_sp.to_sql('bolsa_familia_sp_clean', con=engine, index=False, if_exists='replace')

34267102

In [ ]:
# Passo extra: gerar tabela calendário para o Power BI

# Define o intervalo de datas (jan/25 a mar/26)
data_inicio = '2025-01-01'
data_fim = '2026-03-31'

# Cria a série de datas dia a dia
# freq='D', significa que será dia(day) por dia
datas = pd.date_range(start=data_inicio, end=data_fim, freq='D')

# Cria o DataFrame da dCalendario
df_calendario = pd.DataFrame({'data': datas})

# Gera as colunas derivadas
df_calendario['ano'] = df_calendario['data'].dt.year
df_calendario['mes_num'] = df_calendario['data'].dt.month
df_calendario['mes_nome'] = df_calendario['data'].dt.strftime('%B')
df_calendario['trimestre'] = df_calendario['data'].dt.quarter

# Cria a coluna mes/ano e a coluna de ordenação
df_calendario['mes_ano'] = df_calendario['data'].dt.strftime('%b/%Y')
df_calendario['ordem_mes_ano'] = df_calendario['data'].dt.strftime('%Y%m').astype(int)

# Tradução dos meses
meses_ptbr = {
    'January': 'Janeiro', 'February': 'Fevereiro', 'March': 'Março', 'April': 'Abril',
    'May': 'Maio', 'June': 'Junho', 'July': 'Julho', 'August': 'Agosto',
    'September': 'Setembro', 'October': 'Outubro', 'November': 'Novembro', 'December': 'Dezembro'
}
df_calendario['mes_nome'] = df_calendario['mes_nome'].map(meses_ptbr)

# Exporta para csv
df_calendario.to_csv('dCalendario_bolsa_familia.csv', sep=';', index=False, encoding='latin-1')

print('Tabela dCalendario gerada com sucesso!')
display(df_calendario.sample(15))


Tabela dCalendario gerada com sucesso!


,data,ano,mes_num,mes_nome,trimestre,mes_ano,ordem_mes_ano
198,2025-07-18,2025,7,Julho,3,Jul/2025,202507
64,2025-03-06,2025,3,Março,1,Mar/2025,202503
118,2025-04-29,2025,4,Abril,2,Apr/2025,202504
93,2025-04-04,2025,4,Abril,2,Apr/2025,202504
117,2025-04-28,2025,4,Abril,2,Apr/2025,202504
81,2025-03-23,2025,3,Março,1,Mar/2025,202503
247,2025-09-05,2025,9,Setembro,3,Sep/2025,202509
274,2025-10-02,2025,10,Outubro,4,Oct/2025,202510
218,2025-08-07,2025,8,Agosto,3,Aug/2025,202508
319,2025-11-16,2025,11,Novembro,4,Nov/2025,202511
